# Search, Download, and Stream NISAR GCOV Data with `earthaccess`


[`earthaccess`](https://github.com/nsidc/earthaccess) is an open source Python library to search, download, and stream NASA {abbr}`EO (Earth Observation)` data. his notebook demonstrates how to search and download NISAR GCOV data with `earthaccess`

<hr>

## Overview

1. [Prerequisites](earthaccess-prereqs)
1. [Search for `GCOV` data with `earthaccess`](earthaccess-search)
1. [(Option 1) Download the data to a specified directory](earthaccess-download)
1. [(Option 2) Stream the data and load with `xarray`](earthaccess-stream)
1. [Summary](earthaccess-summary)
1. [Resources and references](earthaccess-resources)

<hr>

(earthaccess-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |

- **Rough Notebook Time Estimate**: 3 minutes

<hr>

(earthaccess-search)=
## 2. Search for `GCOV` data with `earthaccess`

:::{tip} NISAR L2 GCOV dataset collection names

Use these collection names for the `short_name` argument in the `earthaccess.search_data()` in the following code cell:
- NISAR_L2_GCOV_BETA_V1
- NISAR_L2_GCOV_PROVISIONAL_V1
- NISAR_L2_GCOV_V1
:::

In [1]:
import earthaccess

earthaccess.login()
results = earthaccess.search_data(
    short_name='NISAR_L2_GCOV_BETA_V1',  
    bounding_box=(-10, 20, 10, 50),
    temporal=("2025-11", "2025-12"),
    # count=10 # (optional) limit the number of results
)
print(f"Found {len(results)} GCOV products:")
results

Found 69 GCOV products:


[Collection: {'ShortName': 'NISAR_L2_GCOV_BETA_V1', 'Version': '1'}
 Spatial coverage: {'GranuleLocalities': ['Land', 'Europe'], 'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 40.06429, 'Longitude': -0.93995}, {'Latitude': 41.93635, 'Longitude': -1.69264}, {'Latitude': 41.21061, 'Longitude': -4.54045}, {'Latitude': 39.36322, 'Longitude': -3.69594}, {'Latitude': 40.06429, 'Longitude': -0.93995}]}}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2025-11-09T05:15:42.000000Z', 'EndingDateTime': '2025-11-09T05:16:13.999342Z'}}
 Size(MB): 1721.1890840530396
 Data: ['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_004_159_A_022_2005_DHDH_A_20251109T051542_20251109T051613_X05010_N_F_J_001/NISAR_L2_PR_GCOV_004_159_A_022_2005_DHDH_A_20251109T051542_20251109T051613_X05010_N_F_J_001.h5'],
 Collection: {'ShortName': 'NISAR_L2_GCOV_BETA_V1', 'Version': '1'}
 Spatial coverage: {'GranuleLocalities': [

<hr>

(earthaccess-download)=
## 3. (Option 1) Download the data to a specified directory

In [2]:
from pathlib import Path

data_dir = Path.home() / "GCOV_data_earthaccess_example"
files = earthaccess.download(results[0], data_dir)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

<hr>

## 4. (Option 2) Stream the data over HTTPS

:::{note} This example was copied from the [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/earthaccess/)

Direct HTTPS access is a streaming option that works when running outside of the AWS `us-west-2` region.
:::

In [3]:
%%time

import earthaccess
import xarray as xr

https_links = results[0].data_links(access='external')

fsspec_config = {
    'cache_type': 'background',
    'block_size': 16*1024*1024,  # 16 MB
}

fs = earthaccess.get_fsspec_https_session()
ds = xr.open_datatree(
   fs.open(https_links[0], **fsspec_config),
   engine='h5netcdf',
   decode_timedelta=False,
   phony_dims="access"
)

ds

CPU times: user 1.85 s, sys: 293 ms, total: 2.14 s
Wall time: 6.77 s


<xarray.DataTree>
Group: /
│   Attributes:
│       Conventions:         CF-1.7
│       contact:             nisar-sds-ops@jpl.nasa.gov
│       institution:         NASA JPL
│       mission_name:        NISAR
│       reference_document:  D-102274 NISAR NASA SDS Product Specification Level-...
│       title:               NISAR L2 GCOV Product
└── Group: /science
    └── Group: /science/LSAR
        ├── Group: /science/LSAR/GCOV
        │   ├── Group: /science/LSAR/GCOV/grids
        │   │   ├── Group: /science/LSAR/GCOV/grids/frequencyA
        │   │   │       Dimensions:                (yCoordinates: 17568, xCoordinates: 17820,
        │   │   │                                   phony_dim_0: 2)
        │   │   │       Coordinates:
        │   │   │         * yCoordinates           (yCoordinates) float64 141kB 4.665e+06 ... 4.314e+06
        │   │   │         * xCoordinates           (xCoordinates) float64 143kB 3.442e+05 ... 7.006e+05
        │   │   │       Dimensions without coordinates: phony_dim_0
        │   │   │       Data variables:
        │   │   │           HHHH                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           HVHV                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           listOfCovarianceTerms  (phony_dim_0) |S4 8B ...
        │   │   │           listOfPolarizations    (phony_dim_0) |S2 4B ...
        │   │   │           mask                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfLooks          (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfSubSwaths      uint8 1B ...
        │   │   │           projection             uint32 4B ...
        │   │   │           rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           xCoordinateSpacing     float64 8B ...
        │   │   │           yCoordinateSpacing     float64 8B ...
        │   │   └── Group: /science/LSAR/GCOV/grids/frequencyB
        │   │           Dimensions:                (yCoordinates: 4392, xCoordinates: 4455,
        │   │                                       phony_dim_1: 2)
        │   │           Coordinates:
        │   │             * yCoordinates           (yCoordinates) float64 35kB 4.665e+06 ... 4.314e+06
        │   │             * xCoordinates           (xCoordinates) float64 36kB 3.442e+05 ... 7.005e+05
        │   │           Dimensions without coordinates: phony_dim_1
        │   │           Data variables:
        │   │               HHHH                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               HVHV                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               listOfCovarianceTerms  (phony_dim_1) |S4 8B ...
        │   │               listOfPolarizations    (phony_dim_1) |S2 4B ...
        │   │               mask                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               numberOfLooks          (yCoordinates, xCoordinates) float32 78MB ...
        │   │               numberOfSubSwaths      uint8 1B ...
        │   │               projection             uint32 4B ...
        │   │               rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 78MB ...
        │   │               xCoordinateSpacing     float64 8B ...
        │   │               yCoordinateSpacing     float64 8B ...
        │   └── Group: /science/LSAR/GCOV/metadata
        │       ├── Group: /science/LSAR/GCOV/metadata/attitude
        │       │       Dimensions:       (phony_dim_2: 94, phony_dim_3: 3, phony_dim_4: 4)
        │       │       Dimensions without coordinates: phony_dim_2, phony_dim_3, phony_dim_4
        │       │       Data variables:
        │       │           attitudeType  |S10 10B ...
        │       │           eulerAngles   (phony_dim_2, phony_dim_3) float64 2kB ...
        │       │           quaternions   (phony_dim_2, phony_dim_4) float64 3kB ...
        │   

(earthaccess-s3-stream)=
## 5. (Option 3) Stream the data from S3 and load with `xarray`

:::{note} This example was copied from the [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/earthaccess/)

Direct S3 access only works when running in the AWS `us-west-2` region.
:::

#### Notes on Chunking

This notebook utilizes chunking to break large arrays into smaller blocks. Chunking allows users to inspect and process subsets of a dataset without loading entire arrays into memory at once. In this example, data are accessed using an 8 MB block size for cloud access.

For additional benchmarking and recommendations for NISAR direct-access workflows, see Henry Rodman's [Reading NISAR granules directly from S3](https://gist.github.com/hrodmn/5a377531c19da6f6d2c2bb149c450b3d).

In [4]:
%%time

import earthaccess
import xarray as xr

s3_links = results[0].data_links(access='direct')

fsspec_config = {
    'cache_type': 'background',
    'block_size': 8*1024*1024,  # 8 MB
}

# The endpoint must be set to the NISAR-specific endpoint
endpoint = 'https://nisar.asf.earthdatacloud.nasa.gov/s3credentials'
fs = earthaccess.get_s3_filesystem(endpoint=endpoint)
ds = xr.open_datatree(
   fs.open(s3_links[0], **fsspec_config),
   engine='h5netcdf',
   decode_timedelta=False,
   phony_dims="access"
)

ds

CPU times: user 1.03 s, sys: 233 ms, total: 1.26 s
Wall time: 2.1 s


<xarray.DataTree>
Group: /
│   Attributes:
│       Conventions:         CF-1.7
│       contact:             nisar-sds-ops@jpl.nasa.gov
│       institution:         NASA JPL
│       mission_name:        NISAR
│       reference_document:  D-102274 NISAR NASA SDS Product Specification Level-...
│       title:               NISAR L2 GCOV Product
└── Group: /science
    └── Group: /science/LSAR
        ├── Group: /science/LSAR/GCOV
        │   ├── Group: /science/LSAR/GCOV/grids
        │   │   ├── Group: /science/LSAR/GCOV/grids/frequencyA
        │   │   │       Dimensions:                (yCoordinates: 17568, xCoordinates: 17820,
        │   │   │                                   phony_dim_0: 2)
        │   │   │       Coordinates:
        │   │   │         * yCoordinates           (yCoordinates) float64 141kB 4.665e+06 ... 4.314e+06
        │   │   │         * xCoordinates           (xCoordinates) float64 143kB 3.442e+05 ... 7.006e+05
        │   │   │       Dimensions without coordinates: phony_dim_0
        │   │   │       Data variables:
        │   │   │           HHHH                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           HVHV                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           listOfCovarianceTerms  (phony_dim_0) |S4 8B ...
        │   │   │           listOfPolarizations    (phony_dim_0) |S2 4B ...
        │   │   │           mask                   (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfLooks          (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           numberOfSubSwaths      uint8 1B ...
        │   │   │           projection             uint32 4B ...
        │   │   │           rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB ...
        │   │   │           xCoordinateSpacing     float64 8B ...
        │   │   │           yCoordinateSpacing     float64 8B ...
        │   │   └── Group: /science/LSAR/GCOV/grids/frequencyB
        │   │           Dimensions:                (yCoordinates: 4392, xCoordinates: 4455,
        │   │                                       phony_dim_1: 2)
        │   │           Coordinates:
        │   │             * yCoordinates           (yCoordinates) float64 35kB 4.665e+06 ... 4.314e+06
        │   │             * xCoordinates           (xCoordinates) float64 36kB 3.442e+05 ... 7.005e+05
        │   │           Dimensions without coordinates: phony_dim_1
        │   │           Data variables:
        │   │               HHHH                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               HVHV                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               listOfCovarianceTerms  (phony_dim_1) |S4 8B ...
        │   │               listOfPolarizations    (phony_dim_1) |S2 4B ...
        │   │               mask                   (yCoordinates, xCoordinates) float32 78MB ...
        │   │               numberOfLooks          (yCoordinates, xCoordinates) float32 78MB ...
        │   │               numberOfSubSwaths      uint8 1B ...
        │   │               projection             uint32 4B ...
        │   │               rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 78MB ...
        │   │               xCoordinateSpacing     float64 8B ...
        │   │               yCoordinateSpacing     float64 8B ...
        │   └── Group: /science/LSAR/GCOV/metadata
        │       ├── Group: /science/LSAR/GCOV/metadata/attitude
        │       │       Dimensions:       (phony_dim_2: 94, phony_dim_3: 3, phony_dim_4: 4)
        │       │       Dimensions without coordinates: phony_dim_2, phony_dim_3, phony_dim_4
        │       │       Data variables:
        │       │           attitudeType  |S10 10B ...
        │       │           eulerAngles   (phony_dim_2, phony_dim_3) float64 2kB ...
        │       │           quaternions   (phony_dim_2, phony_dim_4) float64 3kB ...
        │   

<hr>

(earthaccess-summary)=
## 5. Summary

You now have the tools and knowledge that you need to search, download, and stream data using the `earthaccess` Python package.

<hr>

(earthaccess-resources)=
## 6. Resources and references
 - [earthaccess](https://earthaccess.readthedocs.io/en/stable/)
 
**Author:** Alex Lewandowski

Streaming examples copied directly from the [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/earthaccess/)